# Training data generation

In [1]:
from src import llms, utils, dataset_gen
import json

TRAINING_DOCS_FILEPATH = './results/training_docs.jsonl'
TRAINING_CANDIDATES_FILEPATH = './results/training_candidates.jsonl'

## Task conflict simulator

In [ ]:
conflict_simulator_llm = llms.OpenAiClient(model='gpt-4o-mini', temperature=0.8)
training_documents = dataset_gen.generate_conflict_context_dataset(conflict_simulator_llm, 100)

# Persist to file
utils.save_dataset_to_jsonl(training_documents, TRAINING_DOCS_FILEPATH)
print(f'Saved {len(training_documents)} docs to the training dataset.')

Saved 100 docs to the training dataset.


## Candidate generation

In [2]:
# Re-load documents from file.
# This is practical if only part of the notebook needs to be run.
training_documents = utils.load_dataset_from_jsonl(TRAINING_DOCS_FILEPATH)
print(f"Loaded {len(training_documents)} documents")

Loaded 100 documents


In [ ]:
agent_llm = llms.OpenAiClient(model="gpt-4.1-mini", temperature=0.8)  # For generating interventions

interventions_for_training_data = dataset_gen.generate_intervention_candidates_dataset(
    documents=training_documents,
    agent_llm_1=agent_llm,
    agent_llm_2=agent_llm,
    use_hidden_prompt_for_interventions=True,
    conversation_points=[2, 3, 4],
    use_chain_of_thought=True
)

print(f"\n\nGenerated {len(interventions_for_training_data)} training examples")



Generated 12 training examples


In [ ]:
with open(TRAINING_CANDIDATES_FILEPATH, 'w', encoding='utf-8') as f:
    for item in interventions_for_training_data:
        f.write(item.to_json() + '\n')


## Judge labeling of candidates

In [2]:
interventions_for_training_data = []

# Re-load documents from file.
# This is practical if only part of the notebook needs to be run.

training_documents = utils.load_dataset_from_jsonl(TRAINING_DOCS_FILEPATH)
print(f"Loaded {len(training_documents)} documents")

with open(TRAINING_CANDIDATES_FILEPATH, 'r', encoding='utf-8') as f:
    for line in f:
        interventions_for_training_data.append(json.loads(line.strip()))
print(f"Loaded {len(interventions_for_training_data)} intervention pairs")



Loaded 100 documents
Loaded 12 intervention pairs


In [ ]:

judge_llm = llms.OpenAiClient(model="gpt-4.1", temperature=0.4) 

training_data = dataset_gen.generate_judge_dataset(
    intervention_pairs=interventions_for_training_data,
    context=training_documents,
    judge_llm=judge_llm,
    judge_method='selection',
    use_chain_of_thought=True
)

print(f"\n\nGenerated {len(training_data)} training examples")

# Save training data
with open('results/training_data.jsonl', 'w', encoding='utf-8') as f:
    for item in training_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

Using 2 comments
 * Intervention 1: {
  "reasoning": "The peer suggests adding specific evidence to strengthen the opponents' argument, while the author wants to preserve the emotional and community perspective without turning the sentence into a researched argument. A compromise can incorporate a mild hint of evidence without overly formalizing the statement, thus respecting the author's intent and addressing the peer's desire for more concreteness.",
  "intervention_type": "compromise",
  "comment": "I suggest we change this sentence to: \"Opponents of the renovation argue that the changes will lead to a significant reduction in the park's natural habitat, which could threaten the diverse species of birds and small mammals that inhabit the area, based on concerns raised in similar urban development projects.\" This version maintains the community's emotional viewpoint while subtly introducing a nod to external evidence, satisfying both the author's and peer's intentions."
}
 * Interv